In [1]:
import os
import pandas as pd
from fuzzywuzzy import process

# === CONFIG ===
ward_reference_file = "official_wards.csv"  # from OS Boundary-Line
input_files = "input_files"
output_files = "processed_files"
match_threshold = 85  # fuzzy match confidence


# === HELPERS ===
def normalize_name(name):
    if not isinstance(name, str):
        return ""
    name = (
        name.lower()
        .replace("&", "and")
        .replace("-", " ")
        .replace("ward", "")
        .replace("council", "")
        .strip()
    )
    return " ".join(name.split())


def best_match(name, ref_list, threshold):
    """Return best fuzzy match if above threshold."""
    if not name:
        return None, 0
    # ensure ref_list is a Python list
    ref_list = list(ref_list)
    result = process.extractOne(name, ref_list)
    if result is None:
        return None, 0
    # unpack dynamically: first two items
    match = result[0]
    score = result[1]
    if score >= threshold:
        return match, score
    else:
        return None, score

    
input_dir = "scotland_data_code_attempt"
output_dir = "processed_files"
match_threshold = 85  # fuzzy match confidence

In [2]:
# Import ward file (filter for Scotland!)
wards_ref = pd.read_csv('wardcodelist.csv')
wards_ref["clean_name"] = wards_ref["ward_name"].apply(normalize_name)
wards_ref = wards_ref.drop(columns=['WD25NMW','ObjectId'])
wards_ref = wards_ref[wards_ref['ward_code'].str.startswith("S", na=False)]

In [5]:
file = 'raw_census_data/uv501_education.csv'
converting_db = pd.read_csv(file)
converting_db.head()

for idx, row in converting_db.iterrows():
    clean_title = normalize_name(row["Electoral Ward 2022"])
    match, score = best_match(clean_title, wards_ref["clean_name"], match_threshold)
    if match:
        ward_code = wards_ref.loc[wards_ref["clean_name"] == match, "ward_code"].iloc[0]
        converting_db.at[idx, "Electoral Ward 2022"] = ward_code
converting_db["pct_uni_educated"] = round(converting_db["Degree level qualifications or above"]/converting_db["All people aged 16 and over"],4)
converting_db=converting_db[["Electoral Ward 2022","pct_uni_educated"]]
converting_db.to_csv(file, index=False)